In [1]:
# mamba activate spatialdata_env
import tifffile
import os
from spatialdata.models import Image2DModel
import spatialdata_io
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import numpy as np
from spatialdata.models import ShapesModel
import anndata
import scanpy as sc
from spatialdata import SpatialData
from spatialdata import polygon_query
from spatialdata import bounding_box_query
# from napari_spatialdata import Interactive
from spatialdata.models import TableModel
from spatialdata_plot.pl.utils import set_zero_in_cmap_to_transparent
from spatialdata import SpatialData
from napari.utils.colormaps import Colormap
from napari_spatialdata import Interactive
import spatialdata as sd
from scipy import ndimage as ndi
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib as mpl
from spatialdata.transformations import (
    BaseTransformation,
    Identity,
    Sequence,
    align_elements_using_landmarks,
    get_transformation,
    set_transformation,
    Affine
)

/vf/users/kanferg/conda_v1/envs/spatialdata_env/lib/python3.12/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/vf/users/kanferg/conda_v1/envs/spatialdata_env/lib/python3.12/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
/vf/users/kanferg/conda_v1/envs/spatialdata_env/lib/python3.12/site-packages/anndata/__init__.py:44: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead

In [ ]:
path_orig = '/data/HiTIF/data/spatialomics/neuroblastoma/data/SCAF4480/PrimaryAnalysisOutput/SCAF4480_PA_xenium/'
sdata = sd.read_zarr(path_orig + 'output-XETG00202__0059709_Left__SCAF04480_Left_R1__20250403__144627.zarr')

In [ ]:
# Interactive(sdata)

In [ ]:
path_zarr = '/data/HiTIF/data/spatialomics/neuroblastoma/data/SCAF4480/PrimaryAnalysisOutput/SCAF4480_PA_xenium/nb_orig_label_v1.zarr'
sdata.write(path_zarr,overwrite=True)

In [2]:
path_zarr = '/data/HiTIF/data/spatialomics/neuroblastoma/data/SCAF4480/PrimaryAnalysisOutput/SCAF4480_PA_xenium/nb_orig_label_v1.zarr'
sdata = sd.read_zarr(path_zarr)

version mismatch: detected: RasterFormatV02, requested: FormatV04
/vf/users/kanferg/conda_v1/envs/spatialdata_env/lib/python3.12/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/vf/users/kanferg/conda_v1/envs/spatialdata_env/lib/python3.12/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/vf/users/kanferg/conda_v1/envs/spatialdata_env/lib/python3.12/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/vf/users/kanferg/conda_v1/envs/spatialdata_env/lib/python3.12/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/vf/users/kanferg/conda_v1/envs/spatia

In [ ]:
# Interactive(sdata)

In [3]:
plygoneList = [l for l in sdata.shapes.keys() if l not in ('cell_boundaries')]

In [4]:
from tqdm import tqdm
sdata['table'].obs['sample_id'] = 'remove'
df = sdata['table'].obs
for pg in tqdm(plygoneList):
    polygon = sdata.shapes[pg].geometry.iloc[0]
    ind = polygon_query(sdata['cell_boundaries'],polygon,target_coordinate_system="global").index.tolist()
    df.loc[df['cell_id'].isin(ind),'sample_id'] = pg
    del ind

100%|███████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [05:00<00:00,  6.01s/it]


In [5]:
andata = sdata['table']

In [11]:
andata = andata[~andata.obs['sample_id'].isin(['remove'])]

In [10]:
pathout = '/data/HiTIF/data/spatialomics/neuroblastoma/data/andata_orig'
andata.write_h5ad(os.path.join(pathout, "andata_nb_orig.h5ad"))

In [2]:
path_zarr = '/data/HiTIF/data/spatialomics/neuroblastoma/data/SCAF4480/PrimaryAnalysisOutput/SCAF4480_PA_xenium/nb_orig_label_v1.zarr'
sdata = sd.read_zarr(path_zarr)

version mismatch: detected: RasterFormatV02, requested: FormatV04
/vf/users/kanferg/conda_v1/envs/spatialdata_env/lib/python3.12/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/vf/users/kanferg/conda_v1/envs/spatialdata_env/lib/python3.12/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/vf/users/kanferg/conda_v1/envs/spatialdata_env/lib/python3.12/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/vf/users/kanferg/conda_v1/envs/spatialdata_env/lib/python3.12/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/vf/users/kanferg/conda_v1/envs/spatia

In [3]:
Interactive(sdata)

2026-06-09 12:51:04.982 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2026-06-09 12:51:04.983 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2026-06-09 12:51:48.764 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2026-06-09 12:51:48.768 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
